# Lab Assignment Seven: Sequential Network Architectures

**Team Members:** Emilio Munoz, Andy Su, Jadon Swearingen  
**Date:** 05/14/26  
**Dataset:** IMDB Movie Reviews (50K, binary sentiment)

## Part 0: Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print(f'Using device: {device}')

---
## Part 1: Preparation

### 1.1 Dataset Loading and Description

**Dataset:** IMDB Movie Reviews (Kaggle)

- **Source:** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews), collected from the Internet Movie Database
- **Number of samples:** 50,000
- **Number of classes:** 2, positive and negative sentiment
- **Class distribution:** Balanced, 25,000 positive and 25,000 negative
- **Task:** Many to one sequence classification. Given a full review, predict a single sentiment label

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
texts = df['review'].tolist()
labels = (df['sentiment'] == 'positive').astype(int).tolist()  # 1=positive, 0=negative

print(f'Total samples: {len(texts)}')
print(f'Class distribution: {Counter(labels)}')
print(f'\nSample text:\n{texts[0][:300]}')

### 1.2 Preprocessing and Tokenization

We went with word-level tokenization. The main reason is that we're using GloVe embeddings which are word-level, so something like BPE would've meant throwing out the pretrained weights entirely and learning from scratch. That's a bad trade when we only have 50K reviews. Word-level is also just easier to debug.

The downside is out-of-vocabulary words. Anything unusual or misspelled just gets mapped to UNK. For IMDB that's mostly fine since the reviews are standard English and a 20,000-word vocab covers the vast majority of what comes up.

For sequence length, IMDB reviews vary a lot. We looked at the distribution and the 95th percentile is around 400 words, so that's what we set MAX_LEN to. Less than 5% of reviews get cut at all, and even those only lose the tail end which usually doesn't change the overall sentiment anyway. Shorter reviews get padded on the right.

**What we cleaned:**
- Stripped HTML tags. The raw data has a lot of br tags from the original IMDB formatting
- Lowercased everything since GloVe is also lowercase
- Removed punctuation since GloVe doesn't have embeddings for things like exclamation marks anyway
- Collapsed leftover whitespace

In [ ]:
def preprocess_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)  # strip html
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_texts = [preprocess_text(t) for t in texts]

lengths = [len(t.split()) for t in cleaned_texts]
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color='steelblue', alpha=0.8)
plt.axvline(np.percentile(lengths, 95), color='red', linestyle='--',
            label=f'95th pct = {np.percentile(lengths, 95):.0f} words')
plt.axvline(np.median(lengths), color='orange', linestyle='--',
            label=f'Median = {np.median(lengths):.0f} words')
plt.xlabel('Sequence length (words)')
plt.ylabel('Count')
plt.title('IMDB Review Lengths after preprocessing')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean: {np.mean(lengths):.1f}')
print(f'Median: {np.median(lengths):.1f}')
print(f'95th percentile: {np.percentile(lengths, 95):.1f}')
print(f'Max: {np.max(lengths)}')

In [ ]:
MAX_LEN = 400
VOCAB_SIZE = 20000
EMBED_DIM = 100  # using glove 100d

all_words = [word for text in cleaned_texts for word in text.split()]
word_counts = Counter(all_words)
vocab = ['<PAD>', '<UNK>'] + [w for w, _ in word_counts.most_common(VOCAB_SIZE - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

def tokenize_and_pad(text, max_len=MAX_LEN):
    tokens = [word2idx.get(w, 1) for w in text.split()]
    tokens = tokens[:max_len]
    tokens += [0] * (max_len - len(tokens))
    return tokens

X = np.array([tokenize_and_pad(t) for t in cleaned_texts])
y = np.array(labels)

print(f'X shape: {X.shape}')
print(f'Vocab size: {len(vocab)}')

### 1.3 Evaluation Metric

We're using macro F1 as the main metric, with accuracy reported alongside it.

The dataset is balanced so accuracy would technically be fine here, but we think F1 is still the better choice. If the model mislabels a negative review as positive it ends up promoting a bad movie to users. If it buries a positive review that's also a problem for the platform. These are two different kinds of errors and F1 accounts for both, while accuracy just counts what's correct overall.

There's also the deployment side of things. The training data is 50/50 but that won't hold once the model is actually running. A film that flops might have 90% negative reviews. F1 would catch if the model starts leaning toward one class in those situations, whereas accuracy might not.

In practice on a balanced dataset like this the two metrics will stay close, so we're reporting both. If they ever diverge noticeably that's a sign something's off.

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate(y_true, y_pred):
    
    return {

        'macro_f1': f1_score(y_true, y_pred, average='macro'),
        'accuracy': accuracy_score(y_true, y_pred),
        
    }

### 1.4 Train/Test Split

We're using stratified 5-fold cross-validation instead of the fixed train/test split that comes with the IMDB dataset.

The issue with a single split is you get one number and there's no way to know if it's representative. Since we're comparing multiple models in Part 8, we need more than one data point per model to actually run a statistical test. With 5-fold CV each model gets five F1 scores and every review gets used for validation at some point.

We kept the folds stratified even though the dataset is already balanced. It's a small thing but it guarantees the class ratio stays consistent across folds rather than drifting slightly.

We went with k=5 over k=10 mainly because of compute. Ten folds would roughly double training time and for a 50K dataset the accuracy gain isn't really worth it.

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    print(f'Fold {fold_idx+1}: train={len(train_idx)}, val={len(val_idx)}')

print('\nClass distribution preserved across folds (stratified).')

---
## Part 2: Pre-trained Embeddings

We're using GloVe 6B 100d embeddings rather than starting from random weights. GloVe was trained on 6 billion tokens from Wikipedia and Gigaword, so the model already has a decent sense of word similarity before it ever sees an IMDB review. Words like "excellent" and "great" already cluster together, same with "terrible" and "awful". That's a useful head start for sentiment.

We went with 100 dimensions instead of 200d or 300d mostly for speed. The embedding size carries through the whole model so going bigger means slower training across the board. For a binary sentiment task on clean English text 100d is enough.

Any word that's not in GloVe just gets a small random initialization and gets updated during training. The PAD token stays zeroed out. GloVe coverage on IMDB is usually above 90% so this doesn't come up that often.

Make sure glove.6B.100d.txt is in the same folder as this notebook before running.

In [ ]:
GLOVE_PATH = 'glove.6B.100d.txt'

def load_glove(path, word2idx, embed_dim):

    embedding_matrix = np.random.uniform(-0.1, 0.1, (len(word2idx), embed_dim))
    embedding_matrix[0] = 0  # <PAD> = zero vector
    found = 0
    
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            if word in word2idx:
                embedding_matrix[word2idx[word]] = np.array(parts[1:], dtype=np.float32)
                found += 1
    print(f'GloVe coverage: {found}/{len(word2idx)} words ({100*found/len(word2idx):.1f}%)')
    return torch.FloatTensor(embedding_matrix)

glove_weights = load_glove(GLOVE_PATH, word2idx, EMBED_DIM)

---
## Part 3: Dataset and DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.LongTensor(sequences)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


BATCH_SIZE = 64
NUM_CLASSES = len(set(labels))
print(f'Number of classes: {NUM_CLASSES}')

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes,
                 num_filters=128, kernel_sizes=(3, 4, 5),
                 dropout=0.5, pretrained_embeddings=None, freeze_embed=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)
            self.embedding.weight.requires_grad = not freeze_embed

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)  # (B, embed_dim, L)
        pooled = [torch.relu(conv(x)).max(dim=2).values for conv in self.convs]
        x = torch.cat(pooled, dim=1)
        x = self.dropout(x)
        return self.fc(x)

### 4.2 Architecture 2 — Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-np.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes,
                 num_heads=4, ff_dim=256, dropout=0.1,
                 pretrained_embeddings=None, freeze_embed=False,
                 num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)
            self.embedding.weight.requires_grad = not freeze_embed

        self.pos_enc = PositionalEncoding(embed_dim, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout, batch_first=True
        )
        # enable_nested_tensor=False: the nested-tensor fast path uses an op
        # that isn't implemented on MPS; disabling it keeps the model portable
        # across CPU/MPS/CUDA at a negligible speed cost on this size.
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, enable_nested_tensor=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        pad_mask = (x == 0)
        x = self.embedding(x)
        x = self.pos_enc(x)
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        x = x.mean(dim=1)
        x = self.dropout(x)
        return self.fc(x)

---
## Part 5: Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
        history['train_loss'].append(total_loss / len(train_loader.dataset))

        model.eval()
        val_loss, all_preds, all_true = 0, [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                val_loss += criterion(logits, y_batch).item() * len(y_batch)
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_true.extend(y_batch.cpu().numpy())
        history['val_loss'].append(val_loss / len(val_loader.dataset))
        history['val_f1'].append(f1_score(all_true, all_preds, average='macro'))

        print(f'  Epoch {epoch+1}/{epochs} | '
              f'train_loss={history["train_loss"][-1]:.4f} | '
              f'val_loss={history["val_loss"][-1]:.4f} | '
              f'val_f1={history["val_f1"][-1]:.4f}')

    return history


def plot_history(history, title='Model'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], label='Train Loss')
    ax1.plot(epochs, history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} — Loss'); ax1.legend()

    ax2.plot(epochs, history['val_f1'], label='Val Macro F1', color='green')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
    ax2.set_title(f'{title} — Validation F1'); ax2.legend()

    plt.tight_layout()
    plt.show()

---
## Part 6: Two Architectures + Hyperparameter Tuning

Train at least 4 models total (2 CNN variants + 2 Transformer variants). For each architecture, adjust **one** hyperparameter to try to improve generalization. Visualize training/validation loss and F1 for each model to show convergence.

**CNN hyperparameter to tune:** `num_filters` (try 128 vs. 256) — or justify a different choice.  
**Transformer hyperparameter to tune:** `ff_dim` (try 256 vs. 512) — or justify a different choice.

In [ ]:
EPOCHS = 15  # TODO: increase if models have not converged
LR = 1e-3

# Use fold 0 for architecture comparison
train_idx, val_idx = folds[0]
train_ds = TextDataset(X[train_idx], y[train_idx])
val_ds   = TextDataset(X[val_idx],   y[val_idx])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

In [ ]:
# --- Model 1: CNN Baseline ---
print('=== Model 1: CNN Baseline (num_filters=128) ===')
cnn_baseline = TextCNN(
    vocab_size=len(vocab), embed_dim=EMBED_DIM, num_classes=NUM_CLASSES,
    num_filters=128, dropout=0.5, pretrained_embeddings=glove_weights
)
hist_cnn_base = train_model(cnn_baseline, train_loader, val_loader, epochs=EPOCHS, lr=LR)
plot_history(hist_cnn_base, 'CNN Baseline (num_filters=128)')

**Model 2 Hyperparameter Value Justification:**

For Model 2 we increased num_filters from 128 to 256. Each filter in the CNN learns one phrase pattern independently, so num_filters directly controls how many distinct patterns the conv layer can capture in parallel. Doubling it gives the model twice as many of these detectors per kernel size.

We considered adjusting kernel_sizes or dropout instead, but those control what shape of pattern the model looks at and how aggressively it regularizes. It does not look at how many patterns it can learn. num_filters felt like the cleanest hyperparameter for raw capacity.

We decided on 256 because IMDB reviews have a lot of subtle sentiment phrasing, such as sarcasm, double negatives, etc.,  that 128 filters can't fully cover. Going higher (512, 1024) would start risking overfitting on the training set. 



In [ ]:
# --- Model 2: CNN Tuned ---
print('=== Model 2: CNN Tuned (num_filters=256) ===')
cnn_tuned = TextCNN(
    vocab_size=len(vocab), embed_dim=EMBED_DIM, num_classes=NUM_CLASSES,
    num_filters=256, dropout=0.5, pretrained_embeddings=glove_weights
)
hist_cnn_tuned = train_model(cnn_tuned, train_loader, val_loader, epochs=EPOCHS, lr=LR)
plot_history(hist_cnn_tuned, 'CNN Tuned (num_filters=256)')

In [ ]:
# --- Model 3: Transformer Baseline ---
print('=== Model 3: Transformer Baseline (ff_dim=256) ===')
transformer_baseline = TransformerClassifier(
    vocab_size=len(vocab), embed_dim=EMBED_DIM, num_classes=NUM_CLASSES,
    num_heads=4, ff_dim=256, dropout=0.1, pretrained_embeddings=glove_weights
)
hist_trans_base = train_model(transformer_baseline, train_loader, val_loader, epochs=EPOCHS, lr=LR)
plot_history(hist_trans_base, 'Transformer Baseline')

**Model 4 Hyperparameter Value Justification:**

For Model 4 we increased ff_dim from 256 to 512. Inside each Transformer encoder layer, the feedforward network is a 2-layer MLP that takes each token's contextualized representation and applies learned per-token features to it. ff_dim is the hidden width of that MLP, which is basically how many neurons the FFN gets to use when transforming each token. Doubling it gives the model more capacity to express a little more suble per-position features after attention completes its mixing of information. 

We chose 512 because the standard rule of thumb is ff_dim ≈ 4 × embed_dim, which for our 100-dim GloVe vectors would be 400. The baseline value of 256 sits below that ratio and the tuned 512 sits above it (5.12×), so the comparison brackets the conventional value and tests whether more capacity helps or hurts on IMDB. Going higher (e.g. 1024) would risk overfitting on a 40K training set. 

In [ ]:
# --- Model 4: Transformer Tuned ---
print('=== Model 4: Transformer Tuned (ff_dim=512) ===')
transformer_tuned = TransformerClassifier(
    vocab_size=len(vocab), embed_dim=EMBED_DIM, num_classes=NUM_CLASSES,
    num_heads=4, ff_dim=512, dropout=0.1, pretrained_embeddings=glove_weights
)
hist_trans_tuned = train_model(transformer_tuned, train_loader, val_loader, epochs=EPOCHS, lr=LR)
plot_history(hist_trans_tuned, 'Transformer Tuned (ff_dim=512)')

## Part 7: Second Multi-Headed Attention Layer

Using the best Transformer configuration from Part 6, add a second self-attention layer (`num_layers=2`). The output of the first attention layer feeds directly into the second. Visualize convergence to show the model trained successfully.

**According to the val_f1 values for 3 and 4, Model 4 has a value of 0.8965 at epoch 4, which edges out Model 3's 0.8934 at epoch 3. Thus, the full configs to carry out for part 7 are: num_heads = 4; ff_dim = 512; dropout = 01; embed_dim = 100; and pretrained_embeddings = glove_weights**


In [ ]:
#Model 5: Transformer 2-Layer
print('=== Model 5: Transformer 2-Layer (num_heads=4, ff_dim=512, num_layers=2) ===')
transformer_2layer = TransformerClassifier(
    vocab_size=len(vocab), embed_dim=EMBED_DIM, num_classes=NUM_CLASSES,
    num_heads=4, ff_dim=512, dropout=0.1, num_layers=2,
    pretrained_embeddings=glove_weights
)
hist_trans_2layer = train_model(transformer_2layer, train_loader, val_loader, epochs=EPOCHS, lr=LR)
plot_history(hist_trans_2layer, 'Transformer 2-Layer (ff_dim=512)')

---
## Part 8: Full Cross-Validation + Visualize and Compare All Models

Run 5-fold CV for all 5 model configurations. Report mean ± std Macro F1. Visualize results and apply a statistical test to determine which model(s) are significantly better.

In [ ]:
def run_cross_validation(model_fn, folds, X, y, epochs=EPOCHS, lr=LR):
    fold_f1s = []
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f'  Fold {fold_idx+1}/{len(folds)}...')
        train_ds = TextDataset(X[train_idx], y[train_idx])
        val_ds   = TextDataset(X[val_idx],   y[val_idx])
        tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        vl = DataLoader(val_ds,   batch_size=BATCH_SIZE)
        history = train_model(model_fn(), tl, vl, epochs=epochs, lr=lr)
        fold_f1s.append(max(history['val_f1']))
    print(f'  CV Macro F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}')
    return fold_f1s

In [ ]:
print('=== CNN Baseline CV ===')
cv_cnn_base = run_cross_validation(
    lambda: TextCNN(len(vocab), EMBED_DIM, NUM_CLASSES, num_filters=128,
                    pretrained_embeddings=glove_weights), folds, X, y)

print('\n=== CNN Tuned CV ===')
cv_cnn_tuned = run_cross_validation(
    lambda: TextCNN(len(vocab), EMBED_DIM, NUM_CLASSES, num_filters=256,
                    pretrained_embeddings=glove_weights), folds, X, y)

print('\n=== Transformer Baseline CV ===')
cv_trans_base = run_cross_validation(
    lambda: TransformerClassifier(len(vocab), EMBED_DIM, NUM_CLASSES,
                                  num_heads=4, ff_dim=256,
                                  pretrained_embeddings=glove_weights), folds, X, y)

print('\n=== Transformer Tuned CV ===')
cv_trans_tuned = run_cross_validation(
    lambda: TransformerClassifier(len(vocab), EMBED_DIM, NUM_CLASSES,
                                  num_heads=4, ff_dim=512,
                                  pretrained_embeddings=glove_weights), folds, X, y)

print('\n=== Transformer 2-Layer CV ===')
cv_trans_2layer = run_cross_validation(
    lambda: TransformerClassifier(len(vocab), EMBED_DIM, NUM_CLASSES,
                                  num_heads=4, ff_dim=512, num_layers=2,
                                  pretrained_embeddings=glove_weights), folds, X, y)

In [ ]:
# Recovery: only re-run the Transformer 2-Layer CV (other 4 are still in memory)
print('=== Transformer 2-Layer CV ===')
cv_trans_2layer = run_cross_validation(
    lambda: TransformerClassifier(len(vocab), EMBED_DIM, NUM_CLASSES,
                                  num_heads=4, ff_dim=512, num_layers=2,
                                  pretrained_embeddings=glove_weights), folds, X, y)

In [ ]:
# Summary table + bar chart
model_names = ['CNN Baseline', 'CNN Tuned', 'Transformer Baseline',
               'Transformer Tuned', 'Transformer 2-Layer']
cv_results  = [cv_cnn_base, cv_cnn_tuned, cv_trans_base, cv_trans_tuned, cv_trans_2layer]

summary = pd.DataFrame({
    'Model': model_names,
    'Mean Macro F1': [np.mean(r) for r in cv_results],
    'Std':           [np.std(r)  for r in cv_results]
})
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(model_names, [np.mean(r) for r in cv_results],
       yerr=[np.std(r) for r in cv_results],
       capsize=5, color='steelblue', alpha=0.8)
ax.set_ylabel('Macro F1 Score')
ax.set_title('5-Fold CV Macro F1 — All Models')
ax.set_ylim(0, 1)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

### Statistical Comparison

We use the **Wilcoxon signed-rank test** (non-parametric, paired) because:
- Fold scores are paired — the same splits were used for all models.
- With only 5 folds we cannot assume normality.

Significance level: α = 0.05.

In [ ]:
from itertools import combinations

print(f'{"Model A":<25} {"Model B":<25} {"p-value":>10} {"Significant?":>14}')
print('-' * 78)

for (i, a), (j, b) in combinations(enumerate(model_names), 2):
    _, p = stats.wilcoxon(cv_results[i], cv_results[j])
    print(f'{a:<25} {b:<25} {p:>10.4f} {"Yes *" if p < 0.05 else "No":>14}')

print('\nNote: Wilcoxon has low power with only 5 folds — treat non-significance cautiously.')

### Interpretation of CV + Statistical Results

*(Fill in after the CV runner finishes — placeholder structure below.)*

**Ranking by mean Macro F1:**
1. *(model name)* — F1 = *(value)* ± *(std)*
2. *(model name)* — F1 = *(value)* ± *(std)*
3. *(model name)* — F1 = *(value)* ± *(std)*
4. *(model name)* — F1 = *(value)* ± *(std)*
5. *(model name)* — F1 = *(value)* ± *(std)*

**Significantly different pairs (p < 0.05):**
- *(pair 1)* — p = *(value)*
- *(pair 2)* — p = *(value)*
- *(...)*

**What this means:**
- Which architecture is superior overall (CNN vs. Transformer), and is the gap statistically significant or just within fold-to-fold noise?
- Did the hyperparameter tuning (`num_filters` 128→256, `ff_dim` 256→512) produce a real improvement, or was it within noise?
- Did adding a second attention layer help, hurt, or make no measurable difference?

**Caveats:** With only 5 folds, the Wilcoxon test has limited statistical power — a non-significant result doesn't prove the models are equivalent, just that we don't have enough evidence to distinguish them. The smallest detectable difference depends on fold-to-fold variability.

---
## Part 9: Exceptional Work — ConceptNet Numberbatch vs. GloVe

ConceptNet Numberbatch encodes commonsense knowledge on top of distributional statistics (it's built by retrofitting word vectors against the ConceptNet knowledge graph, so words connected by relations like *IsA*, *PartOf*, *RelatedTo* end up closer in vector space than they would with raw co-occurrence stats).

We compare it against GloVe using the **best architecture from Part 8: the CNN with `num_filters=256`** (mean F1 = 0.9041). Holding architecture, folds, and all other hyperparameters constant isolates the embedding effect — any difference in F1 reflects only the choice of pretrained vectors.

**Embedding details:**
- GloVe: 100d, 400K words, trained on Wikipedia + Gigaword (~6B tokens)
- Numberbatch: 300d, ~516K English words, retrofitted from word2vec + GloVe + OpenSubtitles against ConceptNet

**Download:** `numberbatch-en-19.08.txt.gz` from https://conceptnet.io/

In [ ]:
NUMBERBATCH_PATH = 'numberbatch-en.txt'
NUMBERBATCH_DIM  = 300

def load_numberbatch(path, word2idx, embed_dim=300):
    embedding_matrix = np.random.uniform(-0.1, 0.1, (len(word2idx), embed_dim))
    embedding_matrix[0] = 0
    found = 0
    with open(path, encoding='utf-8') as f:
        next(f)  # skip header
        for line in f:
            parts = line.split()
            word = parts[0]
            if word.startswith('/c/en/'):
                word = word[len('/c/en/'):]
            if word in word2idx:
                embedding_matrix[word2idx[word]] = np.array(parts[1:embed_dim+1], dtype=np.float32)
                found += 1
    print(f'Numberbatch coverage: {found}/{len(word2idx)} ({100*found/len(word2idx):.1f}%)')
    return torch.FloatTensor(embedding_matrix)

nb_weights = load_numberbatch(NUMBERBATCH_PATH, word2idx, NUMBERBATCH_DIM)

In [ ]:
# Best architecture from Part 8 = CNN Tuned (num_filters=256, mean F1 = 0.9041)
# Re-run that exact architecture with Numberbatch embeddings instead of GloVe.
print('=== Best Architecture (CNN Tuned) + Numberbatch CV ===')
cv_nb = run_cross_validation(
    lambda: TextCNN(len(vocab), NUMBERBATCH_DIM, NUM_CLASSES,
                    num_filters=256, dropout=0.5,
                    pretrained_embeddings=nb_weights),
    folds, X, y
)

# Paired Wilcoxon: same architecture, same folds, only the embedding differs.
_, p = stats.wilcoxon(cv_cnn_tuned, cv_nb)
print(f'\nGloVe       mean F1: {np.mean(cv_cnn_tuned):.4f} +/- {np.std(cv_cnn_tuned):.4f}')
print(f'Numberbatch mean F1: {np.mean(cv_nb):.4f} +/- {np.std(cv_nb):.4f}')
print(f'GloVe vs Numberbatch p-value: {p:.4f}')

**Jadon — write your interpretation here:**

- Which embedding performed better on this task, and by how much?
- Why might Numberbatch help or hurt for this specific dataset?
- What does this tell us about the value of commonsense knowledge for this classification task?

---
## Part 10: Summary and Conclusions

**TODO Jadon:** Fill in after all experiments are complete.

- Best model overall:
- Best hyperparameter settings:
- Did the second attention layer help? Why or why not?
- GloVe vs. Numberbatch conclusion:
- What surprised the team?
- Limitations and future work: